# sberchall — Kaggle runner

This notebook is a **stub on purpose**. It clones the repo and runs its entry point:

```
python -m src
```

Nothing else lives here. What a run *does* — the configuration, the stages, the reporting —
is `src/experiment.py` in the repo. So the loop is: commit to `main`, press **Run all**
here, read the log. This notebook should not need to change between experiments.

## Settings (both required)

1. **Accelerator -> GPU** (P100 or T4). Training is not CPU-feasible.
2. **Internet -> On.** The clone is a network call.

The repo is public, so the clone is anonymous — no Kaggle secret, no SSH key, no token.
`J.npy` and `h_train.npy` are tracked in git, so no dataset needs attaching.

Artefacts (`best.pt`, `submission_train.csv`, `summary.json`, ...) are written straight to
`/kaggle/working` by the pipeline and show up in the output pane.

In [ ]:
REPO   = "https://github.com/brkdrd/sberchall.git"
BRANCH = "main"          # point at a branch to try an experiment before merging it

import shutil, subprocess, sys
from pathlib import Path

# Clone outside /kaggle/working: the checkout is not an output, only what src/ writes is.
SRC = Path("/tmp/sberchall")
if SRC.exists():
    shutil.rmtree(SRC)

clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(SRC)],
                       capture_output=True, text=True)
if clone.returncode:
    raise RuntimeError(clone.stderr + "\n\nClone failed. Check, in order:\n"
                       "  1. Settings -> Internet is ON (off gives a DNS/timeout error);\n"
                       "  2. the repo is still public (403 / 'Authentication failed' if not);\n"
                       f"  3. the branch '{BRANCH}' exists.")
print(subprocess.run(["git", "-C", str(SRC), "log", "-1", "--pretty=cloned %h %s"],
                     capture_output=True, text=True).stdout)

# Everything from here on is the repo's code. -u so the log streams into this cell live.
proc = subprocess.Popen([sys.executable, "-u", "-m", "src"], cwd=str(SRC),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc:
    raise RuntimeError(f"python -m src exited with code {rc} (see the log above)")
print("\n[runner] done")